In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

buyer_profile = pd.read_csv('../data/processed/buyer_profile_dataset.csv')

buyer_profile.head()

,client_id,total_properties_owned,total_investment_value,average_property_value,average_floor_area,age,client_type,gender,country,region,acquisition_purpose,loan_applied,referral_channel,satisfaction_score
0,C0001,4,1246764.72,311691.180,983.885,58.0,Individual,F,Usa,California,Home,Yes,Website,4.0
1,C0003,5,1661457.59,332291.518,1058.110,67.0,Individual,M,Usa,California,Home,Yes,Agency,4.0
2,C0006,5,1514131.06,302826.212,989.590,69.0,Individual,M,Usa,California,Home,Yes,Website,3.0
3,C0009,5,1820832.35,364166.470,1305.644,51.0,Individual,M,Usa,California,Investment,No,Agency,5.0
4,C0013,5,2142750.89,428550.178,1407.376,64.0,Individual,M,Usa,California,Home,Yes,Website,1.0


In [2]:
buyer_profile.info()

buyer_profile.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 855 entries, 0 to 854
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   client_id               855 non-null    str    
 1   total_properties_owned  855 non-null    int64  
 2   total_investment_value  855 non-null    float64
 3   average_property_value  855 non-null    float64
 4   average_floor_area      855 non-null    float64
 5   age                     855 non-null    float64
 6   client_type             855 non-null    str    
 7   gender                  855 non-null    str    
 8   country                 855 non-null    str    
 9   region                  855 non-null    str    
 10  acquisition_purpose     855 non-null    str    
 11  loan_applied            855 non-null    str    
 12  referral_channel        855 non-null    str    
 13  satisfaction_score      855 non-null    float64
dtypes: float64(5), int64(1), str(8)
memory usage: 129.3 K

client_id                 0
total_properties_owned    0
total_investment_value    0
average_property_value    0
average_floor_area        0
age                       0
client_type               0
gender                    0
country                   0
region                    0
acquisition_purpose       0
loan_applied              0
referral_channel          0
satisfaction_score        0
dtype: int64

In [3]:
numeric_cols = [
    'age',
    'satisfaction_score',
    'total_properties_owned',
    'total_investment_value',
    'average_property_value',
    'average_floor_area'
]

for col in numeric_cols:
    buyer_profile[col] = buyer_profile[col].fillna(
        buyer_profile[col].median()
    )

In [4]:
buyer_profile['client_type_corp'] = (
    buyer_profile['client_type']
    .str.lower()
    .eq('corporate')
    .astype(int)
)

buyer_profile['investment_purpose'] = (
    buyer_profile['acquisition_purpose']
    .str.lower()
    .eq('investment')
    .astype(int)
)

buyer_profile['loan_flag'] = (
    buyer_profile['loan_applied']
    .str.lower()
    .eq('yes')
    .astype(int)
)

In [5]:
buyer_profile['investment_intensity'] = (
    buyer_profile['total_investment_value'] /
    buyer_profile['total_properties_owned']
)

buyer_profile['portfolio_size_score'] = (
    buyer_profile['total_properties_owned'] *
    buyer_profile['average_property_value']
)

buyer_profile['investment_density'] = (
    buyer_profile['total_investment_value'] /
    buyer_profile['average_floor_area']
)

In [6]:
buyer_profile['engagement_score'] = (
    buyer_profile['satisfaction_score'] *
    buyer_profile['total_properties_owned']
)

In [7]:
clustering_features = [
    'age',
    'satisfaction_score',
    'total_properties_owned',
    'total_investment_value',
    'average_property_value',
    'average_floor_area',
    'client_type_corp',
    'investment_purpose',
    'loan_flag',
    'investment_intensity',
    'portfolio_size_score',
    'investment_density',
    'engagement_score'
]

feature_matrix = buyer_profile[clustering_features]

feature_matrix.head()

,age,satisfaction_score,total_properties_owned,total_investment_value,average_property_value,average_floor_area,client_type_corp,investment_purpose,loan_flag,investment_intensity,portfolio_size_score,investment_density,engagement_score
0,58.0,4.0,4,1246764.72,311691.180,983.885,0,0,1,311691.180,1246764.72,1267.185413,16.0
1,67.0,4.0,5,1661457.59,332291.518,1058.110,0,0,1,332291.518,1661457.59,1570.212539,20.0
2,69.0,3.0,5,1514131.06,302826.212,989.590,0,0,1,302826.212,1514131.06,1530.058974,15.0
3,51.0,5.0,5,1820832.35,364166.470,1305.644,0,1,0,364166.470,1820832.35,1394.585622,25.0
4,64.0,1.0,5,2142750.89,428550.178,1407.376,0,0,1,428550.178,2142750.89,1522.514872,5.0


In [8]:
import joblib
import os

scaler = StandardScaler()

scaled_features = scaler.fit_transform(feature_matrix)

scaled_df = pd.DataFrame(
    scaled_features,
    columns=clustering_features
)

# Create models directory if it does not exist
os.makedirs('../models', exist_ok=True)

# Save the scaler
joblib.dump(scaler, '../models/scaler.pkl')

print('Scaler saved successfully.')

scaled_df.head()

Scaler saved successfully.


,age,satisfaction_score,total_properties_owned,total_investment_value,average_property_value,average_floor_area,client_type_corp,investment_purpose,loan_flag,investment_intensity,portfolio_size_score,investment_density,engagement_score
0,0.158469,0.695625,0.438277,-0.057429,-0.536014,-0.756269,0.0,-0.673856,1.296245,-0.536014,-0.057429,0.648616,0.830775
1,0.681825,0.695625,1.679095,1.158342,-0.245256,-0.426312,0.0,-0.673856,1.296245,-0.245256,1.158342,1.858588,1.508878
2,0.798126,-0.023552,1.679095,0.726419,-0.661136,-0.730908,0.0,-0.673856,1.296245,-0.661136,0.726419,1.698257,0.661249
3,-0.248586,1.414802,1.679095,1.625587,0.204633,0.674068,0.0,1.483997,-0.771459,0.204633,1.625587,1.157318,2.356506
4,0.507373,-1.461906,1.679095,2.569368,1.113357,1.126304,0.0,-0.673856,1.296245,1.113357,2.569368,1.668134,-1.034008


In [9]:
scaled_df.to_csv(
    '../data/processed/clustering_feature_matrix.csv',
    index=False
)

print('Clustering feature matrix saved successfully.')

print(scaled_df.shape)

Clustering feature matrix saved successfully.
(855, 13)


In [10]:
feature_summary = pd.DataFrame({
    'Feature': clustering_features,
    'Mean': scaled_df.mean(),
    'Std': scaled_df.std()
})

feature_summary

,Feature,Mean,Std
age,age,-4.986265e-17,1.000585
satisfaction_score,satisfaction_score,1.454327e-17,1.000585
total_properties_owned,total_properties_owned,-1.163462e-16,1.000585
total_investment_value,total_investment_value,9.972530e-17,1.000585
average_property_value,average_property_value,4.279877e-16,1.000585
average_floor_area,average_floor_area,-1.932178e-16,1.000585
client_type_corp,client_type_corp,0.000000e+00,0.000000
investment_purpose,investment_purpose,-8.206561e-17,1.000585
loan_flag,loan_flag,0.000000e+00,1.000585
investment_intensity,investment_intensity,4.175997e-16,1.000585
